In [1]:
pip install faker pandas numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\agusb\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from faker import Faker

In [3]:
fake = Faker('es_ES')
Faker.seed(
    42
)
random.seed(42)
np.random.seed(42)

# Tabla vehicles

In [4]:
# Definimos las reglas de negocio lógicas para los vehículos de FleetLogix
vehicle_configs = [
    {
        'type': 'Camión Grande',
        'min_cap': 8000.0,
        'max_cap': 15000.0,
        'fuels': ['Diesel'],
    },
    {
        'type': 'Camión Mediano',
        'min_cap': 3500.0,
        'max_cap': 7999.0,
        'fuels': ['Diesel', 'Gasolina'],
    },
    {
        'type': 'Van',
        'min_cap': 800.0,
        'max_cap': 3499.0,
        'fuels': ['Diesel', 'Gasolina', 'Eléctrico'],
    },
    {
        'type': 'Motocicleta',
        'min_cap': 50.0,
        'max_cap': 250.0,
        'fuels': ['Gasolina', 'Eléctrico'],
    },
]

vehicles = []
used_plates = set()

In [5]:
# Función para asegurar que las patentes sean únicas
def generate_unique_plate():
    while True:
        # fake.bothify mezcla letras (?) y números (#) al azar
        plate = fake.bothify(text='??###??').upper()
        if plate not in used_plates:
            used_plates.add(plate)
            return plate

In [6]:
# 3. Bucle para generar exactamente los 200 registros pedidos
for i in range(1, 201):
    config = random.choice(vehicle_configs)
    vehicle_type = config['type']

In [7]:
# Capacidad y combustible coherentes según el tipo de vehículo
capacity_kg = round(
    random.uniform(config['min_cap'], config['max_cap']), 2
)
fuel_type = random.choice(config['fuels'])

In [8]:
# Faker genera una fecha de adquisición realista en los últimos 5 años
acquisition_date = fake.date_between(
        start_date='-5y', end_date='-1y'
    ).strftime('%Y-%m-%d')

In [9]:
# Estados con probabilidades lógicas (la mayoría activos)
status = np.random.choice(
        ['active', 'maintenance', 'inactive'], p=[0.85, 0.10, 0.05]
    )

vehicles.append({
    'vehicle_id': i,
    'license_plate': generate_unique_plate(),
    'vehicle_type': vehicle_type,
    'capacity_kg': capacity_kg,
    'fuel_type': fuel_type,
    'acquisition_date': acquisition_date,
    'status': status,
    })

In [10]:
# 4. Lo convertimos a un DataFrame de pandas y lo guardamos en CSV
df_vehicles = pd.DataFrame(vehicles)
df_vehicles.to_csv('vehicles.csv', index=False)

print('¡Tabla vehicles generada con éxito!')
print(df_vehicles.head(10))

¡Tabla vehicles generada con éxito!
   vehicle_id license_plate   vehicle_type  capacity_kg fuel_type  \
0         200       PO104IV  Camión Grande      8354.12    Diesel   

  acquisition_date  status  
0       2024-07-15  active  


# Tabla Drivers

In [11]:
drivers = []
used_employee_codes = set()
used_license_numbers = set()

In [12]:
# Rango de fechas coherente para las contrataciones (últimos 5 años)
start_hire_date = datetime(2021, 1, 1)
end_hire_date = datetime(2026, 6, 1)

In [13]:
# 2. Bucle para generar exactamente los 400 registros de conductores
for i in range(1, 401):

  # Generar código de empleado único
  while True:
    employee_code = f'EMP-{random.randint(1000, 9999)}'
    if employee_code not in used_employee_codes:
      used_employee_codes.add(employee_code)
      break

  # Generar número de licencia único
  while True:
    license_number = f'LIC-{random.randint(10000000, 99999999)}'
    if license_number not in used_license_numbers:
      used_license_numbers.add(license_number)
      break

In [14]:
#nombre y apellidos con faker
first_name = fake.first_name() 
last_name = fake.last_name()

In [15]:
# Fecha de contratación aleatoria actualizada
hire_date = start_hire_date + timedelta(
      days=random.randint(0, (end_hire_date - start_hire_date).days)
  )

In [16]:
# Consistencia temporal: La licencia expira entre 2 y 5 años DESPUÉS de ser contratado
license_expiry = hire_date + timedelta(days=random.randint(730, 1825))

In [17]:
# Teléfono de contacto simulado
phone = f'+54911{random.randint(10000000, 99999999)}'

In [18]:
# Estado del conductor (la mayoría activos)
status = np.random.choice(['active', 'inactive'], p=[0.90, 0.10])

drivers.append({
      'driver_id': i,
      'employee_code': employee_code,
      'first_name': first_name,
      'last_name': last_name,
      'license_number': license_number,
      'license_expiry': license_expiry.strftime('%Y-%m-%d'),
      'phone': phone,
      'hire_date': hire_date.strftime('%Y-%m-%d'),
      'status': status,
  })

In [19]:
# 3. Convertir a DataFrame y guardar en CSV
df_drivers = pd.DataFrame(drivers)
df_drivers.to_csv('drivers.csv', index=False)

print('¡Tabla drivers generada con éxito y adaptada al 2026!')
print(df_drivers.head(10))

¡Tabla drivers generada con éxito y adaptada al 2026!
   driver_id employee_code first_name last_name license_number license_expiry  \
0        400      EMP-3344   Cristian     Bonet   LIC-41163805     2027-01-04   

            phone   hire_date    status  
0  +5491171682508  2023-08-03  inactive  


# Tabla Routes

In [20]:
# Fijar semilla para reproducibilidad
random.seed(42)
np.random.seed(42)

In [21]:
# 1. Definir las 5 ciudades principales y distancias base reales aproximadas
cities = ['Buenos Aires', 'Córdoba', 'Rosario', 'Mendoza', 'San Miguel de Tucumán']

city_distances = {
    ('Buenos Aires', 'Córdoba'): (700, 8.5, 4500),
    ('Buenos Aires', 'Rosario'): (300, 3.5, 2000),
    ('Buenos Aires', 'Mendoza'): (1050, 13.0, 6500),
    ('Buenos Aires', 'San Miguel de Tucumán'): (1250, 15.0, 7500),
    ('Córdoba', 'Rosario'): (400, 4.5, 2500),
    ('Córdoba', 'Mendoza'): (600, 7.5, 3500),
    ('Córdoba', 'San Miguel de Tucumán'): (550, 7.0, 3000),
    ('Rosario', 'Mendoza'): (850, 10.0, 5000),
    ('Rosario', 'San Miguel de Tucumán'): (950, 11.5, 5500),
    ('Mendoza', 'San Miguel de Tucumán'): (980, 12.0, 6000),
}

routes = []
route_id = 1

In [22]:
# 2. Generar combinaciones de rutas entre las ciudades hasta completar 50 registros
for c1 in cities:
  for c2 in cities:
    if c1 != c2 and route_id <= 50:
      pair = (c1, c2) if (c1, c2) in city_distances else (c2, c1)
      base_dist, base_dur, base_toll = city_distances[pair]

In [23]:
# Variación realista del 5% al 10% por ruta
dist = round(base_dist * random.uniform(0.95, 1.05), 2)
dur = round(base_dur * random.uniform(0.95, 1.05), 2)
toll = round(base_toll * random.uniform(0.90, 1.10), 2)

routes.append({
          'route_id': route_id,
          'route_code': f'RTE-{100+route_id}',
          'origin_city': c1,
          'destination_city': c2,
          'distance_km': dist,
          'estimated_duration_hours': dur,
          'toll_cost': toll,
})
route_id += 1

In [24]:
# 3. Convertir a DataFrame y guardar en CSV
df_routes = pd.DataFrame(routes)
df_routes.to_csv('routes.csv', index=False)

print('¡Tabla routes generada con éxito (50 registros)!')
print(df_routes.head(10))

¡Tabla routes generada con éxito (50 registros)!
   route_id route_code            origin_city       destination_city  \
0         1    RTE-101  San Miguel de Tucumán  San Miguel de Tucumán   

   distance_km  estimated_duration_hours  toll_cost  
0       993.66                     11.43    5730.04  


In [ ]:
# Tabla Deliveries

In [ ]:
random.seed(42)
np.random.seed(42)

# 1. Función para simular los picos de salida de camiones por hora del día
def get_hourly_distribution():
    weights = [
        0.01, 0.01, 0.01, 0.01, 0.02, 0.04,  # 00-05
        0.09, 0.12, 0.11, 0.08, 0.06, 0.05,  # 06-11
        0.04, 0.07, 0.08, 0.06, 0.05, 0.04,  # 12-17
        0.03, 0.02, 0.02, 0.01, 0.01, 0.01   # 18-23
    ]
    return np.array(weights) / sum(weights)

# 2. Cargar las tablas maestras con validación de seguridad
df_v = pd.read_csv('vehicles.csv')
df_d = pd.read_csv('drivers.csv')
df_r = pd.read_csv('routes.csv')

# Validación robusta para vehículos y conductores activos (fallback a todos si el filtro falla)
active_vehicle_ids = df_v[df_v['status'].str.lower() != 'inactive']['vehicle_id'].values
if len(active_vehicle_ids) == 0:
    active_vehicle_ids = df_v['vehicle_id'].values

active_driver_ids = df_d[df_d['status'].str.lower() != 'inactive']['driver_id'].values
if len(active_driver_ids) == 0:
    active_driver_ids = df_d['driver_id'].values

route_dict = df_r.set_index('route_id')[['estimated_duration_hours', 'distance_km']].to_dict('index')
v_cap_dict = df_v.set_index('vehicle_id')['capacity_kg'].to_dict()

# Ventana temporal de 2 años históricos (hasta Septiembre 2026)
start_trips = datetime(2024, 10, 1)
end_trips = datetime(2026, 9, 30)
total_days = (end_trips - start_trips).days

hourly_p = get_hourly_distribution()
hours = np.arange(24)

# Muestreo masivo optimizado con numpy para 100,000 viajes
random_days = np.random.randint(0, total_days, size=100000)
selected_hours = np.random.choice(hours, size=100000, p=hourly_p)
selected_minutes = np.random.randint(0, 60, size=100000)

selected_v_ids = np.random.choice(active_vehicle_ids, size=100000)
selected_d_ids = np.random.choice(active_driver_ids, size=100000)
selected_r_ids = np.random.choice(list(route_dict.keys()), size=100000)

trips = []

print("Generando 100,000 viajes históricos...")
for i in range(100000):
    dep_dt = start_trips + timedelta(days=int(random_days[i]), hours=int(selected_hours[i]), minutes=int(selected_minutes[i]))
    
    r_info = route_dict[selected_r_ids[i]]
    est_dur = r_info['estimated_duration_hours']
    dist_km = r_info['distance_km']
    
    # Consistencia temporal: Duración real con factor de tráfico/demoras
    actual_dur = est_dur * random.uniform(0.9, 1.3)
    arr_dt = dep_dt + timedelta(hours=actual_dur)
    
    v_cap = v_cap_dict[selected_v_ids[i]]
    tot_weight = round(v_cap * random.uniform(0.3, 0.95), 2)
    fuel_cons = round((dist_km / 100.0) * random.uniform(12, 32), 2)
    
    trips.append({
        'trip_id': i + 1,
        'vehicle_id': int(selected_v_ids[i]),
        'driver_id': int(selected_d_ids[i]),
        'route_id': int(selected_r_ids[i]),
        'departure_datetime': dep_dt.strftime('%Y-%m-%d %H:%M:%S'),
        'arrival_datetime': arr_dt.strftime('%Y-%m-%d %H:%M:%S'),
        'fuel_consumed_liters': fuel_cons,
        'total_weight_kg': tot_weight,
        'status': 'completed'
    })

df_trips = pd.DataFrame(trips)
df_trips.to_csv('trips.csv', index=False)
print(f"-> 'trips.csv' generado con éxito con {len(df_trips)} registros.")
print(df_trips.head(5))

In [ ]:
#Tabla Deliveries

In [ ]:
fake = Faker('es_ES')
random.seed(42)
np.random.seed(42)

# 1. Cargar la tabla de trips creada anteriormente
df_trips = pd.read_csv('trips.csv')

deliveries = []
delivery_id = 1

print('Generando 400,000 entregas asociadas a los viajes...')

# Para agilizar el proceso en bucles grandes, convertimos los datos de trips a listas/diccionarios
trip_records = df_trips[['trip_id', 'arrival_datetime', 'total_weight_kg']].to_dict('records')

for trip in trip_records:
    t_id = trip['trip_id']
    arr_dt = datetime.strptime(trip['arrival_datetime'], '%Y-%m-%d %H:%M:%S')
    tot_weight = trip['total_weight_kg']
    
    # Número de entregas por viaje (entre 2 y 6, siendo 4 la más probable)
    num_deliveries = np.random.choice([2, 3, 4, 5, 6], p=[0.10, 0.20, 0.40, 0.20, 0.10])
    
    # Dividir el peso del viaje entre las entregas de forma proporcional
    weights = np.random.dirichlet(np.ones(num_deliveries), size=1)[0]
    delivery_weights = weights * tot_weight
    
    for j in range(num_deliveries):
        # El scheduled_datetime y delivered_datetime ocurren poco después de que llega el viaje
        sched_dt = arr_dt + timedelta(hours=random.uniform(0.5, 12.0))
        del_dt = sched_dt + timedelta(hours=random.uniform(-0.5, 3.0)) # A veces un poco antes o después
        
        # Asegurar consistencia temporal: delivered >= scheduled o cercano
        if del_dt < arr_dt:
            del_dt = arr_dt + timedelta(minutes=random.randint(10, 180))
            
        status = np.random.choice(['delivered', 'failed', 'pending'], p=[0.92, 0.05, 0.03])
        
        deliveries.append({
            'delivery_id': delivery_id,
            'trip_id': t_id,
            'tracking_number': f"TRK-{fake.bothify(text='??########').upper()}",
            'customer_name': fake.name(),
            'delivery_address': fake.street_address(),
            'package_weight_kg': round(delivery_weights[j], 2),
            'scheduled_datetime': sched_dt.strftime('%Y-%m-%d %H:%M:%S'),
            'delivered_datetime': del_dt.strftime('%Y-%m-%d %H:%M:%S'),
            'delivery_status': status,
            'recipient_signature': fake.name() if status == 'delivered' else 'N/A'
        })
        delivery_id += 1

df_deliveries = pd.DataFrame(deliveries)
df_deliveries.to_csv('deliveries.csv', index=False)
print(f"-> 'deliveries.csv' generado con éxito con {len(df_deliveries)} registros.")
print(df_deliveries.head(5))

In [ ]:
#Tabla Maintenance

In [ ]:
random.seed(42)
np.random.seed(42)

# 1. Cargar los viajes generados previamente
df_trips = pd.read_csv('trips.csv')

# Tipos de mantenimiento especificados en la consigna
maint_types = [
    'Cambio de aceite',
    'Revisión de frenos',
    'Cambio de llantas',
    'Mantenimiento general',
    'Revisión de motor',
    'Alineación y balanceo',
]

last_names = [
    'Gómez',
    'Pérez',
    'Rodríguez',
    'Fernández',
    'López',
    'Martínez',
    'González',
    'García',
]

# Seleccionar aproximadamente 1 de cada 20 viajes (100,000 / 20 = 5,000 mantenimientos)
sampled_trips = df_trips.sample(n=5000, random_state=42).sort_values(
    'departure_datetime'
)

maintenance_records = []
print('Generando 5,000 registros de mantenimiento...')

for idx, row in enumerate(sampled_trips.iterrows(), start=1):
  trip_data = row[1]
  v_id = trip_data['vehicle_id']
  arr_dt = datetime.strptime(
      trip_data['arrival_datetime'], '%Y-%m-%d %H:%M:%S'
  )

  # El mantenimiento ocurre poco después de la llegada del viaje (entre 2 y 48 horas después)
  maint_date = arr_dt + timedelta(hours=random.randint(2, 48))
  maint_type = random.choice(maint_types)

  # Costos estimados realistas en moneda local según el tipo de intervención
  cost_map = {
      'Cambio de aceite': random.uniform(40000, 90000),
      'Revisión de frenos': random.uniform(60000, 150000),
      'Cambio de llantas': random.uniform(150000, 400000),
      'Mantenimiento general': random.uniform(100000, 250000),
      'Revisión de motor': random.uniform(200000, 600000),
      'Alineación y balanceo': random.uniform(30000, 70000),
  }
  cost = round(cost_map[maint_type], 2)

  maintenance_records.append({
      'maintenance_id': idx,
      'vehicle_id': int(v_id),
      'trip_id': int(trip_data['trip_id']),
      'maintenance_date': maint_date.strftime('%Y-%m-%d %H:%M:%S'),
      'maintenance_type': maint_type,
      'cost': cost,
      'workshop_name': f'Taller Mecánico {random.choice(last_names)}',
      'description': (
          f'Realizado {maint_type.lower()} preventivo/correctivo tras'
          ' operación de ruta.'
      ),
  })

df_maint = pd.DataFrame(maintenance_records)
df_maint.to_csv('maintenance.csv', index=False)
print(f"-> 'maintenance.csv' generado con éxito con {len(df_maint)} registros.")
print(df_maint.head(5))